# Chapter 4 Empirical Results & Diagnostic Visualizations (Light-Mode Notebook Style)

**Thesis Title:** *A Coarse-to-Fine Semantic Conflict Detection System for Ex-Ante Davao City Ordinances Using Information Retrieval and Natural Language Inference*  
**Authors:** Ralph Paolo Dulce & Yahyah Odin (Ateneo de Davao University)

This notebook generates seven publication-grade, light-mode figures for Chapter 4 (`CS_Undergraduate_Thesis_Template/figs/`) directly from verified JSON benchmarks and manuscript tables. Every visual is designed to expose multi-dimensional trade-offs, variance across validation regimes, operating thresholds, or error profiles that tabular rows alone obscure:
1. `statistical_validation_model_comparison.png` — Stratified vs. Premise-Grouped 5-Fold CV & 8-Domain x 6-Model Heatmap
2. `threshold_operating_curve_confusion.png` — Multi-Threshold Operating Curves (`tau in [0.20, 0.80]`) & Normalized 3-Class Confusion Matrix (`tau* = 0.45`)
3. `ocr_pareto_and_era_degradation.png` — Extraction Accuracy-vs-CER Pareto Frontier (`N = 60`) & Historical Era Degradation Profile (1956–2021)
4. `stage2_screening_and_finetuning_shift.png` — 28-Model Zero-Shot Landscape & 13-Model Supervised Fine-Tuning Shift
5. `edge_latency_scaling_and_bayesian_ppv.png` — 4-Thread CPU Shortlist Scaling / Padding Comparison & Continuous Bayesian PPV Curves
6. `tier2_graph_recovery_profile.png` — Tier 2 Stage-by-Stage Full-PDF Ablation & Isolated V0 vs. Open-Corpus V5 Per-Defect Profile (`N = 6` Drafts, `7` Defects)
7. `tier3_ablation_and_open_corpus_audit.png` — Tier 3 Stage-by-Stage Ablation (Recall vs. Control Clearance) & Synchronized Full-PDF Open-Corpus Audit Profile (`N = 10` PDFs)

In [1]:
# Cell 1: Imports, Light-Mode Styling & Canonical Data Load
import os, json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
import matplotlib.ticker

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 11.0,
    'axes.titleweight': 'bold',
    'axes.labelsize': 9.6,
    'xtick.labelsize': 8.6,
    'ytick.labelsize': 8.6,
    'legend.fontsize': 8.2,
    'figure.titlesize': 13,
    'axes.edgecolor': '#334155',
    'axes.linewidth': 0.85,
    'figure.facecolor': '#FFFFFF',
    'axes.facecolor': '#FAFCFE',
    'savefig.facecolor': '#FFFFFF',
})

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
FIGS_DIR = os.path.join(REPO_ROOT, 'CS_Undergraduate_Thesis_Template', 'figs')
os.makedirs(FIGS_DIR, exist_ok=True)

with open(os.path.join(REPO_ROOT, 'docs', 'benchmarks', 'chapter4_final_empirical_benchmark.json'), 'r', encoding='utf-8') as f:
    ch4_bench = json.load(f)
with open(os.path.join(REPO_ROOT, 'output', 'numbers_registry.json'), 'r', encoding='utf-8') as f:
    num_reg = json.load(f)
with open(os.path.join(REPO_ROOT, 'output', 'tier2_full_ablation_authentic.json'), 'r', encoding='utf-8') as f:
    tier2_abl = json.load(f)
with open(os.path.join(REPO_ROOT, 'output', 'tier2_live_demo_full_audit_all6.json'), 'r', encoding='utf-8') as f:
    tier2_live = json.load(f)
with open(os.path.join(REPO_ROOT, 'output', 'tier3_full_ablation_authentic.json'), 'r', encoding='utf-8') as f:
    tier3_abl = json.load(f)
with open(os.path.join(REPO_ROOT, 'output', 'tier3_live_demo_full_audit_all10.json'), 'r', encoding='utf-8') as f:
    tier3_live = json.load(f)

print(f'Loaded canonical JSONs from {REPO_ROOT}')
print(f'Tier 2 docs: {len(tier2_live["document_level_results"])} | Tier 3 Full-PDF runs: {len(tier3_live)}')

Loaded canonical JSONs from C:\Users\SHRIMP\Documents\thesis-repo
Tier 3 SC cases: 10 | Tier 3 Full-PDF runs: 10


In [2]:
# Cell 2: Figure 4.7 Replacement — Stratified vs. Premise-Grouped CV & 8x6 Domain Heatmap
fig = plt.figure(figsize=(12.4, 9.8), dpi=300)
gs = GridSpec(2, 1, height_ratios=[1.05, 1.15], hspace=0.40)

# --- Panel A: Stratified 5-Fold vs. Premise-Grouped 5-Fold CV Conflict F1 ---
ax1 = fig.add_subplot(gs[0])
models_cv = [
    ('Zero-Shot\nBaseline', 'zero_shot_baseline', 48.33, 12.06),
    ('all-MiniLM\n-L6-v2', None, 83.41, 3.52),
    ('bert-base\n-uncased', 'bert_base_uncased', 85.59, 2.99),
    ('legal-bert\n-base', 'legal_bert_base', 86.58, 4.29),
    ('ModernBERT\n-base', 'modernbert_base', 86.97, 4.39),
    ('PoL-BERT\n-Large', 'pol_bert_large', 89.29, 5.38),
    ('DeBERTa-v3\n-base-NLI', 'deberta_v3_base_nli', 90.08, 4.21),
]
grouped_vals = {
    'zero_shot_baseline': (46.44, 18.33),
    None: (83.62, 9.51),
    'bert_base_uncased': (num_reg['stage2_premise_grouped_cv']['bert_base_uncased']['grouped_conflict_f1_mean'] * 100,
                          num_reg['stage2_premise_grouped_cv']['bert_base_uncased']['grouped_conflict_f1_sd'] * 100),
    'legal_bert_base': (num_reg['stage2_premise_grouped_cv']['legal_bert_base']['grouped_conflict_f1_mean'] * 100,
                        num_reg['stage2_premise_grouped_cv']['legal_bert_base']['grouped_conflict_f1_sd'] * 100),
    'modernbert_base': (num_reg['stage2_premise_grouped_cv']['modernbert_base']['grouped_conflict_f1_mean'] * 100,
                        num_reg['stage2_premise_grouped_cv']['modernbert_base']['grouped_conflict_f1_sd'] * 100),
    'pol_bert_large': (num_reg['stage2_premise_grouped_cv']['pol_bert_large']['grouped_conflict_f1_mean'] * 100,
                       num_reg['stage2_premise_grouped_cv']['pol_bert_large']['grouped_conflict_f1_sd'] * 100),
    'deberta_v3_base_nli': (num_reg['stage2_premise_grouped_cv']['deberta_v3_base_nli']['grouped_conflict_f1_mean'] * 100,
                            num_reg['stage2_premise_grouped_cv']['deberta_v3_base_nli']['grouped_conflict_f1_sd'] * 100),
}

x = np.arange(len(models_cv))
w = 0.34
strat_m = [m[2] for m in models_cv]
strat_sd = [m[3] for m in models_cv]
grp_m = [grouped_vals[m[1]][0] for m in models_cv]
grp_sd = [grouped_vals[m[1]][1] for m in models_cv]

ax1.bar(x - w/2, strat_m, w, yerr=strat_sd, capsize=3.5,
        color='#1D4ED8', edgecolor='#0F172A', linewidth=0.75, alpha=0.90,
        error_kw=dict(ecolor='#1E293B', lw=1.0), label='Stratified 5-Fold CV (N = 350)')
ax1.bar(x + w/2, grp_m, w, yerr=grp_sd, capsize=3.5,
        color='#0D9488', edgecolor='#0F172A', linewidth=0.75, alpha=0.88,
        error_kw=dict(ecolor='#1E293B', lw=1.0), label='Premise-Grouped 5-Fold CV (22 Statutory Clusters, Zero Leakage)')

for idx in range(len(models_cv)):
    ax1.text(x[idx] - w/2, strat_m[idx] + strat_sd[idx] + 1.3, f'{strat_m[idx]:.1f}%',
             ha='center', va='bottom', fontsize=7.8, fontweight='bold' if idx == 6 else 'normal', color='#1E3A8A')
    ax1.text(x[idx] + w/2, grp_m[idx] + grp_sd[idx] + 1.3, f'{grp_m[idx]:.1f}%',
             ha='center', va='bottom', fontsize=7.8, fontweight='bold' if idx == 6 else 'normal', color='#115E59')

ax1.annotate('Zero-leakage retention:\n90.08% -> 89.45% (-0.63%)',
             xy=(6 + w/2, grp_m[6] + grp_sd[6] + 6.5), xytext=(4.85, 108.5),
             ha='center',
             arrowprops=dict(arrowstyle='->', color='#0F172A', lw=1.0, connectionstyle='arc3,rad=-0.12'),
             fontsize=8.3, fontweight='bold', color='#0F172A',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='#ECFDF5', edgecolor='#0D9488', lw=0.9))

ax1.set_ylabel('Conflict F1-Score (%) [Mean +/- 1 SD]')
ax1.set_title('(A) Stratified vs. Zero-Leakage Premise-Grouped 5-Fold Cross-Validation', pad=26)
ax1.set_xticks(x)
ax1.set_xticklabels([m[0] for m in models_cv], fontsize=8.8)
ax1.set_ylim(20, 118)
ax1.grid(axis='y', linestyle='--', alpha=0.45)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.legend(loc='upper center', bbox_to_anchor=(0.5, 1.11), ncol=2, frameon=True, facecolor='#FFFFFF', edgecolor='#CBD5E1')

# --- Panel B: 8 Macro Legal Domains x 6 Primary Models Heatmap ---
ax2 = fig.add_subplot(gs[1])
domains = [
    '3. Public Finance & Appropriations',
    '1. Executive Issuances & Policy',
    '7. Public Utilities & Franchises',
    '5. Education & Academic Institutions',
    '4. Taxation, Tariffs & Revenue',
    '2. Local Government & Boundaries',
    '6. Public Health & Medical Services',
    '8. Statutory Codes & Amendments',
]
heat_models = [
    'Zero-Shot\nBaseline\n(Rank 6.00)',
    'all-MiniLM\n-L6-v2\n(Rank 5.00)',
    'bert-base\n-uncased\n(Rank 4.00)',
    'legal-bert\n-base\n(Rank 3.00)',
    'ModernBERT\n-base\n(Rank 2.00)',
    'DeBERTa-v3\n-base-NLI\n(Rank 1.00)',
]
heat_matrix = np.array([
    [0.5786, 0.7258, 0.7976, 0.8141, 0.8333, 0.8691],
    [0.6033, 0.7316, 0.7986, 0.8164, 0.8557, 0.8812],
    [0.6179, 0.7558, 0.8227, 0.8349, 0.8649, 0.8761],
    [0.6180, 0.7461, 0.8012, 0.8203, 0.8653, 0.8865],
    [0.6274, 0.7512, 0.8313, 0.8478, 0.8662, 0.8887],
    [0.6344, 0.7581, 0.8271, 0.8367, 0.8682, 0.9021],
    [0.6344, 0.7755, 0.8319, 0.8614, 0.8741, 0.9052],
    [0.6305, 0.7749, 0.8309, 0.8536, 0.8774, 0.9087],
])

im = ax2.imshow(heat_matrix, cmap='YlGnBu', aspect='auto', vmin=0.56, vmax=0.92)
ax2.set_xticks(np.arange(len(heat_models)))
ax2.set_xticklabels(heat_models, fontsize=8.8)
ax2.set_yticks(np.arange(len(domains)))
ax2.set_yticklabels(domains, fontsize=8.8)
ax2.set_title(r'(B) Macro $F_1$-Score Matrix Across Eight Legal Domains (Friedman $\chi_F^2 = 40.00$, $p = 1.49 \times 10^{-7}$)', pad=10)

for i in range(heat_matrix.shape[0]):
    for j in range(heat_matrix.shape[1]):
        val = heat_matrix[i, j]
        txt_color = '#FFFFFF' if val >= 0.79 else '#0F172A'
        ax2.text(j, i, f'{val:.4f}', ha='center', va='center',
                 fontsize=8.6, fontweight='bold' if j == 5 else 'normal', color=txt_color)

rect = patches.Rectangle((4.5, -0.5), 1.0, len(domains), fill=False, edgecolor='#D97706', linewidth=2.2)
ax2.add_patch(rect)

cbar = fig.colorbar(im, ax=ax2, fraction=0.025, pad=0.02)
cbar.set_label('Domain Macro F1-Score', fontsize=9)

fig.text(0.5, 0.012,
         r'Omnibus Friedman Test (8 Domains, df = 5): $\chi_F^2 = 40.0000$ ($p < 0.001$)  |  '
         'Post-hoc Wilcoxon vs. bert-base & legal-bert: W = 0.0 (p = 0.0078)  |  '
         'McNemar vs. Zero-Shot (N = 53): p = 0.0008***',
         ha='center', va='center', fontsize=8.8, color='#1E293B',
         bbox=dict(boxstyle='round,pad=0.35', facecolor='#F8FAFC', edgecolor='#94A3B8', lw=0.8))

out1 = os.path.join(FIGS_DIR, 'statistical_validation_model_comparison.png')
plt.savefig(out1, dpi=300, bbox_inches='tight')
plt.close(fig)
print('Saved:', out1)


Saved: C:\Users\SHRIMP\Documents\thesis-repo\CS_Undergraduate_Thesis_Template\figs\statistical_validation_model_comparison.png


In [3]:
# Cell 3: Threshold Operating Curve (tau in [0.20, 0.80]) & 3-Class Confusion Matrix (tau* = 0.45)
taus = np.array([0.20, 0.30, 0.40, 0.42, 0.45, 0.50, 0.60, 0.70, 0.80])
prec = np.array([69.0, 79.9, 89.7, 91.3, 92.9, 94.3, 95.7, 96.5, 97.5])
rec  = np.array([97.3, 95.5, 93.8, 93.8, 92.9, 88.4, 80.4, 73.2, 68.8])
f1_c = np.array([80.7, 87.0, 91.7, 92.5, 92.9, 91.2, 87.4, 83.2, 80.6])
fn_cnt = np.array([3, 5, 7, 7, 8, 13, 22, 30, 35])
fp_cnt = np.array([49, 27, 12, 10, 8, 6, 4, 3, 2])

fig = plt.figure(figsize=(13.2, 5.8), dpi=300)
outer = GridSpec(1, 2, width_ratios=[1.30, 1.0], wspace=0.38)
left_gs = outer[0].subgridspec(2, 1, height_ratios=[2.2, 1.15], hspace=0.14)

ax_top = fig.add_subplot(left_gs[0])
ax_bot = fig.add_subplot(left_gs[1], sharex=ax_top)
ax_cm  = fig.add_subplot(outer[1])

# Top-left: P / R / F1 curves
ax_top.axvspan(0.40, 0.45, color='#DBEAFE', alpha=0.55, label='Optimal Plateau [0.40, 0.45]')
ax_top.axvline(0.45, color='#1D4ED8', linestyle='--', linewidth=1.3)
ax_top.axvline(0.50, color='#64748B', linestyle=':', linewidth=1.1)

ax_top.plot(taus, prec, marker='o', lw=2.0, color='#059669', label='Precision (%)')
ax_top.plot(taus, rec,  marker='s', lw=2.0, color='#DC2626', label='Recall (%)')
ax_top.plot(taus, f1_c, marker='D', lw=2.3, color='#1D4ED8', label='Contradiction F1 (%)')

ax_top.annotate('tau* = 0.45\nP = R = F1 = 92.9%',
                xy=(0.45, 92.9), xytext=(0.56, 97.6),
                arrowprops=dict(arrowstyle='->', color='#1D4ED8', lw=1.0),
                fontsize=8.2, fontweight='bold', color='#1E3A8A',
                bbox=dict(boxstyle='round,pad=0.25', facecolor='#EFF6FF', edgecolor='#1D4ED8', lw=0.8))
ax_top.text(0.512, 70.0, 'Uncalibrated\ntau = 0.50', fontsize=7.8, color='#475569')

ax_top.set_ylabel('Score (%)')
ax_top.set_title('(A) Threshold Sensitivity Profile (N = 350)', pad=30)
ax_top.set_ylim(64, 104)
ax_top.grid(True, linestyle='--', alpha=0.45)
ax_top.spines['top'].set_visible(False)
ax_top.spines['right'].set_visible(False)
ax_top.tick_params(labelbottom=False)
ax_top.legend(loc='lower center', bbox_to_anchor=(0.5, 1.01), ncol=4, fontsize=7.3, frameon=True, facecolor='#FFFFFF', edgecolor='#CBD5E1')

# Bottom-left: False Negatives vs False Positives across tau
ax_bot.axvspan(0.40, 0.45, color='#DBEAFE', alpha=0.55)
ax_bot.axvline(0.45, color='#1D4ED8', linestyle='--', linewidth=1.3)
ax_bot.axvline(0.50, color='#64748B', linestyle=':', linewidth=1.1)
ax_bot.plot(taus, fn_cnt, marker='^', lw=2.0, color='#B91C1C', label='False Negatives (Missed Conflicts)')
ax_bot.plot(taus, fp_cnt, marker='v', lw=2.0, color='#D97706', label='False Positives (False Alarms)')

ax_bot.text(0.20, 49 + 2.5, '49', ha='center', va='bottom', fontsize=7.9, color='#B45309', fontweight='bold')
ax_bot.text(0.20, 3 + 2.5, '3', ha='center', va='bottom', fontsize=7.9, color='#991B1B', fontweight='bold')
ax_bot.text(0.45, 8 + 3.2, '8 / 8', ha='center', va='bottom', fontsize=7.9, color='#1E3A8A', fontweight='bold')
ax_bot.text(0.50, 13 + 2.5, '13', ha='center', va='bottom', fontsize=7.9, color='#991B1B', fontweight='bold')
ax_bot.text(0.50, 6 - 4.8, '6', ha='center', va='top', fontsize=7.9, color='#B45309', fontweight='bold')
ax_bot.text(0.80, 35 + 2.5, '35', ha='center', va='bottom', fontsize=7.9, color='#991B1B', fontweight='bold')
ax_bot.text(0.80, 2 + 2.5, '2', ha='center', va='bottom', fontsize=7.9, color='#B45309', fontweight='bold')

ax_bot.set_xlabel('Contradiction Decision Threshold (tau*)')
ax_bot.set_ylabel('Error Count')
ax_bot.set_xlim(0.17, 0.83)
ax_bot.set_ylim(-2, 60)
ax_bot.set_xticks([0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80])
ax_bot.grid(True, linestyle='--', alpha=0.45)
ax_bot.spines['top'].set_visible(False)
ax_bot.spines['right'].set_visible(False)
ax_bot.legend(loc='upper center', bbox_to_anchor=(0.5, -0.34), fontsize=7.8, ncol=2, frameon=True, facecolor='#FFFFFF', edgecolor='#CBD5E1')

# Panel B: 3-Class Confusion Matrix at tau* = 0.45
cm = np.array([
    [104,   2,   6],
    [  4, 118,   6],
    [  4,   6, 100],
], dtype=float)
row_pct = cm / cm.sum(axis=1, keepdims=True) * 100.0
labels_cm = ['Contradiction\n(N = 112)', 'Entailment\n(N = 128)', 'Neutral\n(N = 110)']
pred_labels = ['Contradiction', 'Entailment', 'Neutral']

im = ax_cm.imshow(row_pct, cmap='Blues', vmin=0, vmax=100, aspect='equal')
ax_cm.set_xticks(np.arange(3))
ax_cm.set_xticklabels(pred_labels, fontsize=9)
ax_cm.set_yticks(np.arange(3))
ax_cm.set_yticklabels(labels_cm, fontsize=9)
ax_cm.set_xlabel('Predicted Class')
ax_cm.set_ylabel('Actual Gold Class')
ax_cm.set_title('(B) Confusion Matrix at tau* = 0.45 (Acc = 92.0%)', pad=10)

for i in range(3):
    for j in range(3):
        cnt = int(cm[i, j])
        pct = row_pct[i, j]
        txt_col = '#FFFFFF' if pct >= 55 else '#0F172A'
        ax_cm.text(j, i, f'{cnt}\n({pct:.1f}%)', ha='center', va='center',
                   fontsize=9.5, fontweight='bold' if i == j else 'normal', color=txt_col)

cbar = fig.colorbar(im, ax=ax_cm, fraction=0.046, pad=0.04)
cbar.set_label('Row-Normalized Share (%)', fontsize=9)

out2 = os.path.join(FIGS_DIR, 'threshold_operating_curve_confusion.png')
plt.savefig(out2, dpi=300, bbox_inches='tight')
plt.close(fig)
print('Saved:', out2)


Saved: C:\Users\SHRIMP\Documents\thesis-repo\CS_Undergraduate_Thesis_Template\figs\threshold_operating_curve_confusion.png


In [4]:
# Cell 4: Document Extraction Pareto Frontier & Historical Era Degradation (1956-2021)
fig = plt.figure(figsize=(13.2, 5.5), dpi=300)
gs = GridSpec(1, 2, width_ratios=[1.05, 1.05], wspace=0.28)

# --- Panel A: Extraction Accuracy vs. CER Pareto Frontier ---
ax1 = fig.add_subplot(gs[0])
extractors = [
    # name, cer, monetary_acc, legal_acc, table_teds, color, marker, xy_text
    ('Gemini Flash 3.8 (Cloud / Paid)',     2.15, 98.30, 97.20, 94.10, '#DC2626', 'D', (5.4, 98.0)),
    ('Qwen2.5-VL-3B NF4 (Offline / Free)', 3.12, 96.70, 95.80, 91.80, '#0284C7', 'o', (5.6, 89.2)),
    ('Surya v0.6+ (Offline / Free)',        8.45, 80.00, 86.50, 76.40, '#059669', 's', (0.8, 76.5)),
    ('Tesseract v5.x (Offline / Free)',    14.82, 63.30, 74.20, 41.20, '#64748B', '^', (6.4, 64.5)),
]

ax1.axvspan(0, 4.5, ymin=0.72, ymax=1.0, color='#ECFDF5', alpha=0.65)
ax1.text(2.25, 101.2, 'Target Sovereign\nLGU Frontier', ha='center', va='bottom',
         fontsize=8.0, fontweight='bold', color='#047857')

ax1.plot([e[1] for e in extractors], [e[2] for e in extractors],
         linestyle='--', color='#94A3B8', linewidth=1.1, zorder=1)

for name, cer, mon, leg, teds, col, mk, xyt in extractors:
    sz = (teds / 100.0) * 300 + 60
    ax1.scatter(cer, mon, s=sz, color=col, marker=mk, edgecolor='#0F172A', linewidth=1.0, zorder=3, alpha=0.92)
    ax1.annotate(f'{name}\nCER={cer:.2f}% | Mon={mon:.1f}% | TEDS={teds:.1f}%',
                 xy=(cer, mon), xytext=xyt,
                 arrowprops=dict(arrowstyle='->', color='#475569', lw=0.9),
                 fontsize=7.6, fontweight='bold' if 'Qwen' in name else 'normal',
                 color='#0F172A',
                 bbox=dict(boxstyle='round,pad=0.25', facecolor='#FFFFFF', edgecolor=col, lw=0.9))

legend_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#0284C7', markeredgecolor='#0F172A', markersize=8, label='100% Offline & Free'),
    Line2D([0], [0], marker='D', color='w', markerfacecolor='#DC2626', markeredgecolor='#0F172A', markersize=7.5, label='Cloud API / Paywall'),
]
ax1.legend(handles=legend_handles, loc='lower left', frameon=True, facecolor='#FFFFFF', edgecolor='#CBD5E1',
           title='Deployment Mode (Size = TEDS)', title_fontsize=7.9, fontsize=7.9)

ax1.set_xlabel('Character Error Rate (CER %, Lower is Better)')
ax1.set_ylabel('Penalty & Monetary Value Accuracy (%)')
ax1.set_title('(A) Extraction Trade-Off Frontier (N = 60 Benchmark)', pad=10)
ax1.set_xlim(0.0, 16.8)
ax1.set_ylim(56.0, 105.5)
ax1.grid(True, linestyle='--', alpha=0.45)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Panel B: Historical Physical Degradation Across Legislative Eras ---
ax2 = fig.add_subplot(gs[1])
eras = ['Ord. 000044-56\n(1956, Carbon)', 'Ord. 000332-10\n(1968, Faded Ink)', 'Ord. 0166-81\n(1981, Hole Punch)', 'Ord. 000667-21\n(2021, Digital)']
x_e = np.arange(len(eras))
vlm_wer = [12.23, 11.06, 19.78, 0.00]
vlm_cer = [7.66,  5.91,  12.87, 0.02]
cloud_wer_idx = [0, 2, 3]
cloud_wer = [1.92, 8.84, 0.00]
cloud_cer = [1.74, 4.99, 0.00]

ax2.axvspan(-0.4, 2.45, color='#FEF2F2', alpha=0.60)
ax2.text(1.02, 21.8, 'Pre-LGC Mechanical Typewriter Era (Quarantine Trigger)',
         ha='center', va='bottom', fontsize=8.0, fontweight='bold', color='#991B1B')

ax2.plot(x_e, vlm_wer, marker='o', lw=2.2, color='#0284C7', label='Local VLM 4-bit NF4 (WER %)')
ax2.plot(x_e, vlm_cer, marker='s', lw=2.0, linestyle='--', color='#0369A1', label='Local VLM 4-bit NF4 (CER %)')
ax2.plot(cloud_wer_idx, cloud_wer, marker='D', lw=1.9, color='#DC2626', label='Cloud Baseline (WER %)')
ax2.plot(cloud_wer_idx, cloud_cer, marker='^', lw=1.7, linestyle='--', color='#B91C1C', label='Cloud Baseline (CER %)')

# Clean non-overlapping value labels
ax2.text(0, 12.23 + 0.7, '12.23%', ha='center', va='bottom', fontsize=7.8, color='#0369A1', fontweight='bold')
ax2.text(0, 7.66 - 1.3,  '7.66%',  ha='center', va='top',    fontsize=7.6, color='#0C4A6E')
ax2.text(1, 11.06 + 0.7, '11.06%', ha='center', va='bottom', fontsize=7.8, color='#0369A1', fontweight='bold')
ax2.text(1, 5.91 + 0.9,  '5.91%',  ha='center', va='bottom', fontsize=7.6, color='#0C4A6E')
ax2.text(2, 19.78 + 0.6, '19.78%', ha='center', va='bottom', fontsize=7.8, color='#0369A1', fontweight='bold')
ax2.text(1.74, 12.87 + 0.5, '12.87%', ha='center', va='bottom', fontsize=7.6, color='#0C4A6E')
ax2.text(2.14, 8.84 + 0.4, '8.84%', ha='left', va='bottom', fontsize=7.5, color='#991B1B')
ax2.text(3, 0.75, '0.00–0.02%', ha='center', va='bottom', fontsize=7.8, color='#0F172A', fontweight='bold')

ax2.annotate('Cloud Reference\nBaseline (1968)', xy=(1, 3.0), xytext=(1.0, 2.0),
             ha='center', fontsize=7.4, color='#7F1D1D',
             bbox=dict(boxstyle='round,pad=0.2', facecolor='#FFFFFF', edgecolor='#FCA5A5', lw=0.7))

ax2.set_ylabel('Transcription Error Rate (%)')
ax2.set_title('(B) Error Divergence Across Historical Eras', pad=10)
ax2.set_xticks(x_e)
ax2.set_xticklabels(eras, fontsize=8.3)
ax2.set_ylim(-1.2, 27.5)
ax2.grid(True, linestyle='--', alpha=0.45)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.legend(loc='upper left', ncol=2, fontsize=7.5, frameon=True, facecolor='#FFFFFF', edgecolor='#CBD5E1')

out3 = os.path.join(FIGS_DIR, 'ocr_pareto_and_era_degradation.png')
plt.savefig(out3, dpi=300, bbox_inches='tight')
plt.close(fig)
print('Saved:', out3)


Saved: C:\Users\SHRIMP\Documents\thesis-repo\CS_Undergraduate_Thesis_Template\figs\ocr_pareto_and_era_degradation.png


In [5]:
# Cell 5: 28-Model Zero-Shot Screening Landscape & 13-Model Supervised Fine-Tuning Shift
fig = plt.figure(figsize=(14.4, 6.6), dpi=300)
gs = GridSpec(1, 2, width_ratios=[1.05, 1.05], wspace=0.58)

# --- Panel A: 28-Model Zero-Shot Screening (N_test = 53) ---
ax1 = fig.add_subplot(gs[0])
zs_models = [
    ('legal-bert-base',     'Legal-Domain',   110, 0.6087, True,  None),
    ('PoL-BERT-Large',      'Legal-Domain',   340, 0.6667, True,  (122, 0.672)),
    ('InLegalBERT',         'Legal-Domain',   110, 0.5652, True,  None),
    ('CaseLaw-BERT',        'Legal-Domain',   110, 0.5455, False, None),
    ('Lawformer',           'Legal-Domain',   110, 0.4762, False, None),
    ('DeBERTa-v3-base-NLI', 'Pre-Aligned NLI', 86, 0.6957, True,  (21, 0.708)),
    ('DeBERTa-v3-large-NLI','Pre-Aligned NLI',435, 0.7200, True,  (135, 0.732)),
    ('deberta-v3-base',     'Pre-Aligned NLI', 86, 0.5217, True,  None),
    ('roberta-large-mnli',  'Pre-Aligned NLI',355, 0.4444, False, (265, 0.395)),
    ('bart-large-mnli',     'Pre-Aligned NLI',406, 0.4348, False, None),
    ('electra-large-disc',  'Pre-Aligned NLI',335, 0.5385, False, None),
    ('bert-base-uncased',   'Pre-Aligned NLI',110, 0.4545, True,  (16, 0.455)),
    ('ModernBERT-base',     'Long-Context',   149, 0.5600, True,  None),
    ('ModernBERT-large',    'Long-Context',   395, 0.6400, True,  (142, 0.628)),
    ('longformer-base',     'Long-Context',   149, 0.4615, False, None),
    ('bigbird-roberta',     'Long-Context',   128, 0.4286, False, None),
    ('nomic-bert-2048',     'Long-Context',   137, 0.4783, False, None),
    ('bge-reranker-v2-m3',  'Cross-Encoders', 568, 0.5833, True,  (175, 0.580)),
    ('bge-reranker-base',   'Cross-Encoders', 278, 0.5000, False, None),
    ('ms-marco-MiniLM',     'Cross-Encoders',  33, 0.3636, False, None),
    ('mdeberta-v3-base',    'Multilingual',    86, 0.4800, True,  None),
    ('xlm-roberta-base',    'Multilingual',   270, 0.3913, False, None),
    ('xlm-roberta-large',   'Multilingual',   550, 0.4286, False, None),
    ('roberta-tagalog',     'Multilingual',   110, 0.3077, False, (160, 0.305)),
    ('all-MiniLM-L6-v2',    'Distill / Edge',  22, 0.3846, True,  (28, 0.398)),
    ('nli-distilroberta',   'Distill / Edge',  82, 0.4167, True,  None),
    ('distilbert-base',     'Distill / Edge',  66, 0.3478, False, None),
    ('albert-base-v2',      'Distill / Edge',  12, 0.3182, False, (18, 0.318)),
]

fam_colors = {
    'Pre-Aligned NLI': '#1D4ED8',
    'Legal-Domain':    '#059669',
    'Long-Context':    '#7C3AED',
    'Cross-Encoders':  '#D97706',
    'Multilingual':    '#DB2777',
    'Distill / Edge':  '#64748B',
}

ax1.axhspan(0.27, 0.45, color='#FEF2F2', alpha=0.55)
ax1.text(12.5, 0.280, 'Affirmative Bias Zone (Conflict F1 < 0.45)', fontsize=7.6, color='#991B1B', fontweight='bold')
ax1.axhline(0.50, color='#475569', linestyle='--', lw=1.0, alpha=0.7)

seen_p = {}
for name, fam, pm, cf1, short, xyt in zs_models:
    cnt = seen_p.get((pm, round(cf1, 2)), 0)
    seen_p[(pm, round(cf1, 2))] = cnt + 1
    pm_j = pm * (1.0 + 0.04 * (cnt - 0.5)) if cnt > 0 else pm
    col = fam_colors[fam]
    if short:
        ax1.scatter(pm_j, cf1, s=75, color=col, marker='o', edgecolor='#0F172A', linewidth=0.9, zorder=4)
    else:
        ax1.scatter(pm_j, cf1, s=55, facecolor='none', edgecolor=col, marker='o', linewidth=1.35, zorder=3)
    if xyt is not None:
        ax1.annotate(name, xy=(pm_j, cf1), xytext=xyt,
                     arrowprops=dict(arrowstyle='->', color='#475569', lw=0.75),
                     fontsize=7.5, fontweight='bold' if 'DeBERTa' in name else 'normal', color='#0F172A')

ax1.set_xscale('log')
ax1.set_xticks([15, 30, 60, 110, 200, 350, 550])
ax1.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
ax1.set_xlim(10, 700)
ax1.set_ylim(0.27, 0.77)
ax1.set_xlabel('Parameter Count (Millions, Log Scale)')
ax1.set_ylabel('Zero-Shot Conflict F1-Score')
ax1.set_title('(A) Zero-Shot Screening Across 28 Encoders (N_test = 53)', pad=12)
ax1.grid(True, linestyle='--', alpha=0.42)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

fam_handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markeredgecolor='#0F172A', markersize=6.8, label=f)
               for f, c in fam_colors.items()]
fam_handles += [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#334155', markeredgecolor='#0F172A', markersize=6.8, label='Shortlisted (13*)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='none', markeredgecolor='#334155', markersize=6.8, label='Excluded (15)'),
]
ax1.legend(handles=fam_handles, loc='upper center', bbox_to_anchor=(0.5, -0.14),
           fontsize=7.4, ncol=4, frameon=True, facecolor='#FFFFFF', edgecolor='#CBD5E1')

# --- Panel B: Zero-Shot -> Supervised Fine-Tuning Shift (13 Shortlisted Models) ---
ax2 = fig.add_subplot(gs[1])
ft_pairs = [
    ('DeBERTa-v3-large-NLI (435M)', 0.7200, 0.9000),
    ('DeBERTa-v3-base-NLI (86M)',   0.6957, 0.8947),
    ('deberta-v3-base (86M)',       0.5217, 0.8750),
    ('ModernBERT-large (395M)',     0.6400, 0.8750),
    ('PoL-BERT-Large (340M)',       0.6667, 0.8696),
    ('legal-bert-base (110M)',      0.6087, 0.8571),
    ('bge-reranker-v2-m3 (568M)',   0.5833, 0.8571),
    ('bert-base-uncased (110M)',    0.4545, 0.8519),
    ('ModernBERT-base (149M)',      0.5600, 0.8500),
    ('mdeberta-v3-base (86M)',      0.4800, 0.8500),
    ('InLegalBERT (110M)',          0.5652, 0.8333),
    ('nli-distilroberta (82M)',     0.4167, 0.8182),
    ('all-MiniLM-L6-v2 (22M)',      0.3846, 0.8108),
]

y = np.arange(len(ft_pairs))
for idx, (nm, zs, ft) in enumerate(ft_pairs):
    col = '#1D4ED8' if 'DeBERTa-v3-base-NLI' in nm else ('#0D9488' if 'DeBERTa' in nm else '#475569')
    lw = 2.4 if 'DeBERTa-v3-base-NLI' in nm else 1.5
    ax2.plot([zs, ft], [idx, idx], color=col, linewidth=lw, alpha=0.85, zorder=2)
    ax2.scatter(zs, idx, s=55, color='#FFFFFF', edgecolor='#DC2626', linewidth=1.4, zorder=3)
    ax2.scatter(ft, idx, s=65, color=col, marker='D', edgecolor='#0F172A', linewidth=0.8, zorder=4)
    ax2.text(ft + 0.012, idx, f'{ft:.4f} (+{ft - zs:.2f})',
             va='center', ha='left', fontsize=7.8,
             fontweight='bold' if 'DeBERTa-v3' in nm else 'normal',
             color='#1E3A8A' if 'DeBERTa-v3-base-NLI' in nm else '#1E293B')

ax2.set_yticks(y)
ax2.set_yticklabels([p[0] for p in ft_pairs], fontsize=8.2)
ax2.invert_yaxis()
ax2.set_xlabel('Conflict F1-Score')
ax2.set_xlim(0.34, 1.07)
ax2.set_title('(B) Supervised Fine-Tuning Shift (N_val = 52)', pad=12)
ax2.grid(axis='x', linestyle='--', alpha=0.45)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

shift_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#FFFFFF', markeredgecolor='#DC2626', markeredgewidth=1.4, markersize=7, label='Zero-Shot (N_test = 53)'),
    Line2D([0], [0], marker='D', color='w', markerfacecolor='#1D4ED8', markeredgecolor='#0F172A', markersize=7, label='Fine-Tuned (N_val = 52)'),
]
ax2.legend(handles=shift_handles, loc='upper center', bbox_to_anchor=(0.5, -0.14),
           ncol=2, fontsize=8.0, frameon=True, facecolor='#FFFFFF', edgecolor='#CBD5E1')

out4 = os.path.join(FIGS_DIR, 'stage2_screening_and_finetuning_shift.png')
plt.savefig(out4, dpi=300, bbox_inches='tight')
plt.close(fig)
print('Saved:', out4)


Saved: C:\Users\SHRIMP\Documents\thesis-repo\CS_Undergraduate_Thesis_Template\figs\stage2_screening_and_finetuning_shift.png


In [6]:
# Cell 6: Edge Latency Scaling / Padding Optimization & Continuous Bayesian PPV Curves
fig = plt.figure(figsize=(13.2, 5.6), dpi=300)
gs = GridSpec(1, 2, width_ratios=[1.0, 1.08], wspace=0.28)

# --- Panel A: Host A 4-Thread CPU Latency Scaling & Padding Ablation ---
ax1 = fig.add_subplot(gs[0])
ks = np.array([5, 10, 15, 25])
pairs = np.array([35, 70, 105, 175])
times = np.array([4.56, 9.07, 12.68, 24.45])

ax1.axvspan(10, 15, color='#ECFDF5', alpha=0.65, label='Interactive Drafting Zone (k in [10, 15])')
ax1.axhline(15.0, color='#DC2626', linestyle='--', linewidth=1.3, label='15.0 s Interactive Ceiling')

ax1.plot(ks, times, marker='o', markersize=7.5, lw=2.3, color='#1D4ED8', label='Dynamic Sequence Padding')
for k_v, p_v, t_v in zip(ks, pairs, times):
    dy = 1.2 if k_v != 15 else -2.3
    ax1.text(k_v, t_v + dy, f'{t_v:.2f}s\n({p_v} pairs)', ha='center',
             va='bottom' if dy > 0 else 'top', fontsize=8.0, fontweight='bold' if k_v == 15 else 'normal', color='#0F172A')

ax1.errorbar([15], [28.15], yerr=[0.83], fmt='s', markersize=8, capsize=5,
             color='#D97706', markeredgecolor='#0F172A', lw=1.5, label='Static 512-Token Padding (N=30: 28.15 +/- 0.83 s)')
ax1.errorbar([15], [12.69], yerr=[0.37], fmt='D', markersize=7.5, capsize=5,
             color='#059669', markeredgecolor='#0F172A', lw=1.5, zorder=5)
ax1.annotate('-54.9% Runtime\n(Wilcoxon p = 1.73e-6)\nRAM: 1.42 / 8.00 GB',
             xy=(15, 20.6), xytext=(4.5, 21.8),
             arrowprops=dict(arrowstyle='<->', color='#B45309', lw=1.1),
             fontsize=8.0, fontweight='bold', color='#1E293B',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFFBEB', edgecolor='#D97706', lw=0.9))

ax1.set_xlabel('Stage 1 Candidate Shortlist Depth (k)')
ax1.set_ylabel('Full-Document Audit Latency (s) [4-Thread CPU]')
ax1.set_title('(A) Edge CPU Scaling & Padding Speedup (14-Sec. Draft)', pad=10)
ax1.set_xticks(ks)
ax1.set_xlim(3, 27.5)
ax1.set_ylim(0, 38.5)
ax1.grid(True, linestyle='--', alpha=0.45)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.legend(loc='upper left', fontsize=7.6, frameon=True, facecolor='#FFFFFF', edgecolor='#CBD5E1')

# --- Panel B: Bayesian Positive Predictive Value (PPV) Across Prevalence p ---
ax2 = fig.add_subplot(gs[1])
p_grid = np.logspace(-3, np.log10(0.35), 400)

def bayes_ppv(sens, spec, p):
    return (sens * p) / (sens * p + (1.0 - spec) * (1.0 - p)) * 100.0

t2_v5 = tier2_abl['full_pdf_variants']['V5_full_system']['summary']['binary_metrics']
t2_v6 = tier2_abl['full_pdf_variants']['V6_hybrid_kg_expansion']['summary']['binary_metrics']
sens_v5 = t2_v5['contradiction_recall']
spec_v5 = t2_v5['section_level_specificity_pct'] / 100.0
sens_v6 = t2_v6['contradiction_recall']
spec_v6 = t2_v6['section_level_specificity_pct'] / 100.0

regimes = [
    (f'1. 1-Hop Stage 1 KG Exp. V6 (Sens {sens_v6*100:.1f}%, Spec {spec_v6*100:.1f}%)', sens_v6, spec_v6, '#DC2626', '--'),
    (f'2. Hybrid Open-Corpus + KG tau* V4/V5 (Sens {sens_v5*100:.1f}%, Spec {spec_v5*100:.1f}%)', sens_v5, spec_v5, '#1D4ED8', '-'),
    ('3. High-Specificity Mode tau*=0.60 (Sens 80.4%, Spec 98.3%)', 0.8040, 0.9832, '#059669', '-'),
]

for lbl, se, sp, col, ls in regimes:
    ppv = bayes_ppv(se, sp, p_grid)
    ax2.plot(p_grid * 100.0, ppv, linestyle=ls, linewidth=2.2, color=col, label=lbl)
    p1 = bayes_ppv(se, sp, 0.01)
    ax2.scatter([1.0], [p1], s=55, color=col, edgecolor='#0F172A', zorder=5)

p1_r1 = bayes_ppv(sens_v6, spec_v6, 0.01)
p1_r2 = bayes_ppv(sens_v5, spec_v5, 0.01)
p1_r3 = 32.58

ax2.axvline(1.0, color='#334155', linestyle=':', linewidth=1.3)
ax2.axvline(32.0, color='#64748B', linestyle='-.', linewidth=1.1)

ax2.annotate(
    f'LGU Base Rate (p = 1.0%):\n'
    f'Regime 3: {p1_r3:.2f}% PPV (~1 in 3 flags)\n'
    f'Regime 2: {p1_r2:.2f}% PPV (~1 in 10 flags)\n'
    f'Regime 1: {p1_r1:.2f}% PPV (~1 in 16 flags)',
    xy=(1.0, p1_r3), xytext=(0.115, 50.0),
    ha='left', va='bottom',
    arrowprops=dict(arrowstyle='->', color='#059669', lw=1.0, connectionstyle='arc3,rad=0.12'),
    fontsize=7.8, color='#0F172A',
    bbox=dict(boxstyle='round,pad=0.32', facecolor='#F8FAFC', edgecolor='#64748B', lw=0.9)
)

ax2.text(28.5, 14.0, 'Balanced\nBenchmark\n(p = 32.0%)', ha='right', va='bottom', fontsize=7.8, color='#475569', fontweight='bold')

ax2.set_xscale('log')
ax2.set_xticks([0.1, 0.5, 1.0, 5.0, 10.0, 32.0])
ax2.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
ax2.set_xlim(0.1, 36.0)
ax2.set_ylim(0, 114)
ax2.set_xlabel('True Preemption Base-Rate Prevalence p (%, Log Scale)')
ax2.set_ylabel('Projected Positive Predictive Value — PPV (%)')
ax2.set_title('(B) Bayesian Screening PPV Under Base-Rate Shift', pad=10)
ax2.grid(True, which='both', linestyle='--', alpha=0.4)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.legend(loc='upper left', fontsize=7.5, frameon=True, facecolor='#FFFFFF', edgecolor='#CBD5E1')

out5 = os.path.join(FIGS_DIR, 'edge_latency_scaling_and_bayesian_ppv.png')
plt.savefig(out5, dpi=300, bbox_inches='tight')
plt.close(fig)
print('Saved:', out5)

Saved: C:\Users\SHRIMP\Documents\thesis-repo\CS_Undergraduate_Thesis_Template\figs\edge_latency_scaling_and_bayesian_ppv.png


In [7]:
# Cell 7: Tier 2 Stage-by-Stage Full-PDF Ablation & Isolated V0 vs. Open-Corpus V5 Profile
fig = plt.figure(figsize=(14.6, 6.0), dpi=300)
gs = GridSpec(1, 2, width_ratios=[1.12, 1.0], wspace=0.66)

fpv = tier2_abl['full_pdf_variants']
var_specs = [
    ('V1_bm25_only',            'V1: BM25 Only (54p)',                 '#94A3B8'),
    ('V2_minilm_only',          'V2: MiniLM Only (34p)',               '#F59E0B'),
    ('V5_full_system',          'V3-V5: Hybrid + KG tau* + Num (46p)', '#0284C7'),
    ('V6_hybrid_kg_expansion',  'V6: +1-Hop Stage 1 KG (89p)',         '#DC2626'),
]

# --- Panel A: Stage-by-Stage Full-PDF Ablation Metrics ---
ax1 = fig.add_subplot(gs[0])
metric_names = ['Defect\nRecall', 'Section\nSpec.', 'Contra.\nPrec.', 'Conflict\nF1', 'Cost-Sens.\nF2', 'Exact\nTarget (/7)']
x = np.arange(len(metric_names))
bw = 0.20

for i, (vkey, vlbl, col) in enumerate(var_specs):
    sm = fpv[vkey]['summary']
    bm = sm['binary_metrics']
    vals = [
        bm['contradiction_recall'] * 100.0,
        bm['section_level_specificity_pct'],
        bm['contradiction_precision'] * 100.0,
        bm['contradiction_f1'] * 100.0,
        bm['cost_sensitive_f2'] * 100.0,
        (sm['exact_answer_key_statute_matches'] / 7.0) * 100.0,
    ]
    offset = (i - 1.5) * bw
    bars = ax1.bar(x + offset, vals, bw, color=col, edgecolor='#0F172A', linewidth=0.75, label=vlbl, alpha=0.92)
    for b, val in zip(bars, vals):
        ax1.text(b.get_x() + b.get_width() / 2.0, val + 1.8, f'{val:.1f}%',
                 ha='center', va='bottom', rotation=90, fontsize=6.9,
                 fontweight='bold' if vkey == 'V5_full_system' else 'normal', color='#0F172A')

ax1.set_ylabel('Percentage / Score x 100 (%)')
ax1.set_title('(A) Tier 2 Full-PDF Stage-by-Stage Ablation (N = 6 Drafts)', pad=12)
ax1.set_xticks(x)
ax1.set_xticklabels(metric_names, fontsize=8.1)
ax1.set_ylim(0, 126)
ax1.grid(axis='y', linestyle='--', alpha=0.45)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.legend(loc='upper center', bbox_to_anchor=(0.5, -0.14), ncol=2, fontsize=7.5, frameon=True, facecolor='#FFFFFF', edgecolor='#CBD5E1')

# --- Panel B: Per-Defect Isolated V0 vs. Open-Corpus V5 Confidence ---
ax2 = fig.add_subplot(gs[1])
v0_list = tier2_abl['v0_isolated_stage2']['per_defect']
v0_map = {(d['doc_id'], d['sec_num']): d for d in v0_list}
v5_docs = {d['doc_id']: d for d in tier2_abl['full_pdf_variants']['V5_full_system']['documents']}

defect_rows = [
    ('DRAFT-ORD-2026-01', 9,  'DRAFT-01 Sec. 9  (Dog Meat Ban)',        'exact'),
    ('DRAFT-ORD-2026-02', 8,  'DRAFT-02 Sec. 8  (Data Disclosure)',     'exact'),
    ('DRAFT-ORD-2026-03', 4,  'DRAFT-03 Sec. 4  (QRF Diversion)',       'exact'),
    ('DRAFT-ORD-2026-06', 4,  'DRAFT-06 Sec. 4  (45% Agri. Reclass.)',  'exact'),
    ('DRAFT-ORD-2026-01', 13, 'DRAFT-01 Sec. 13 (P50,000 Fine Cap)',    'numeric'),
    ('DRAFT-ORD-2026-04', 8,  'DRAFT-04 Sec. 8  (VAT Exemption)',       'sister'),
    ('DRAFT-ORD-2026-05', 4,  'DRAFT-05 Sec. 4  (Toll Barricades)',     'sister'),
]

y = np.arange(len(defect_rows))
v0_probs, v5_confs, modes = [], [], []
for did, snum, lbl, mode in defect_rows:
    v0_probs.append(float(v0_map[(did, snum)]['best_p_contradiction_pct']))
    doc_pairs = [p for p in v5_docs[did]['conflict_pairs'] if p['sec_num'] == snum]
    v5_confs.append(max(float(p['confidence_pct']) for p in doc_pairs))
    modes.append(mode)

mode_cols = {'exact': '#059669', 'sister': '#0284C7', 'numeric': '#D97706'}
bh = 0.34
ax2.barh(y - bh/2, v0_probs, height=bh, color='#CBD5E1', edgecolor='#334155', linewidth=0.75)
ax2.barh(y + bh/2, v5_confs, height=bh, color=[mode_cols[m] for m in modes], edgecolor='#0F172A', linewidth=0.8)

for idx, (p0, p5, mode) in enumerate(zip(v0_probs, v5_confs, modes)):
    ax2.text(p0 + 1.2, idx - bh/2, f'{p0:.1f}%', va='center', ha='left', fontsize=7.2, color='#334155')
    tag = 'Cap' if mode == 'numeric' else ('Exact' if mode == 'exact' else 'Sister')
    ax2.text(p5 + 1.2, idx + bh/2, f'{p5:.1f}% ({tag})', va='center', ha='left', fontsize=7.4, fontweight='bold', color='#0F172A')

ax2.axvline(45.0, color='#DC2626', linestyle='--', linewidth=1.3)
ax2.set_yticks(y)
ax2.set_yticklabels([r[2] for r in defect_rows], fontsize=8.0)
ax2.invert_yaxis()
ax2.set_xlabel('Contradiction Confidence (%)')
ax2.set_xlim(0, 122)
ax2.set_title('(B) Isolated V0 vs. Open-Corpus V5 Per Defect (N = 7)', pad=12)
ax2.grid(axis='x', linestyle='--', alpha=0.45)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

h2 = [
    patches.Patch(facecolor='#CBD5E1', edgecolor='#334155', label='Isolated V0 P(Contra)'),
    patches.Patch(facecolor='#059669', edgecolor='#0F172A', label='V5 Exact Match (4/7)'),
    patches.Patch(facecolor='#0284C7', edgecolor='#0F172A', label='V5 Sister Law (2/7)'),
    patches.Patch(facecolor='#D97706', edgecolor='#0F172A', label='V5 Numeric Cap (1/7)'),
    Line2D([0], [0], color='#DC2626', linestyle='--', lw=1.3, label='Base tau = 45.0%'),
]
ax2.legend(handles=h2, loc='upper center', bbox_to_anchor=(0.48, -0.14), ncol=2, fontsize=7.2, frameon=True, facecolor='#FFFFFF', edgecolor='#CBD5E1')

out6 = os.path.join(FIGS_DIR, 'tier2_graph_recovery_profile.png')
plt.savefig(out6, dpi=300, bbox_inches='tight')
plt.close(fig)
print('Saved:', out6)

Saved: C:\Users\SHRIMP\Documents\thesis-repo\CS_Undergraduate_Thesis_Template\figs\tier2_graph_recovery_profile.png


In [8]:
# Cell 8: Tier 3 Stage-by-Stage Ablation & Synchronized Full-PDF Open-Corpus Audit (N = 10)
fig = plt.figure(figsize=(14.6, 6.4), dpi=300)
gs = GridSpec(1, 2, width_ratios=[1.05, 1.05], wspace=0.60)

# --- Panel A: Horizontal Stage-by-Stage Ablation (Recall on 5 Invalidated vs. Control Clearance) ---
ax1 = fig.add_subplot(gs[0])
std8 = tier3_abl['standard8_variants']
rev_full = tier3_abl['revcode_full_variants']
rev_sub = tier3_abl['revcode_challenged_sec69_423_variants']
ctrl_ids = ['DAVAO-TIER3-02', 'DAVAO-TIER3-05', 'DAVAO-TIER3-07', 'DAVAO-TIER3-08']

v0_t3 = tier3_abl['v0_isolated_stage2']
v0_st = v0_t3['static_045']

labels_a = [
    'V0: Isolated Target-Law',
    'V1: BM25 Only (185p)',
    'V2: MiniLM Only (215p)',
    'V3: Hybrid Static (171p)',
    'V4: + Real KG tau* (183p)',
    'V5: Full Sys. + Num (189p)',
    'V6: + 1-Hop KG (377p)',
]

inv_recalls = [v0_st['tp'] / 5.0 * 100.0]
ctrl_clears = [v0_st['tn'] / 5.0 * 100.0]
for vkey in ['V1_bm25_only', 'V2_minilm_only', 'V3_hybrid_pure_neural_static', 'V4_hybrid_neural_kg_tau', 'V5_full_system', 'V6_hybrid_kg_expansion']:
    sv = std8[vkey]
    rf = rev_full.get(vkey) or rev_sub.get(vkey)
    hits = 0
    if 5 in sv['DAVAO-TIER3-01']['problematic_sections']: hits += 1
    if 423 in rf['problematic_sections']: hits += 1
    if 5 in sv['DAVAO-TIER3-06']['problematic_sections']: hits += 1
    if any(s in sv['SC-PHIL-09']['problematic_sections'] for s in (1, 4)): hits += 1
    if 1 in sv['SC-PHIL-10']['problematic_sections']: hits += 1
    c_op = sum(sv[c]['substantive_sections'] for c in ctrl_ids)
    c_fl = sum(len(sv[c]['problematic_sections']) for c in ctrl_ids)
    inv_recalls.append(hits / 5.0 * 100.0)
    ctrl_clears.append((c_op - c_fl) / c_op * 100.0)

y1 = np.arange(len(labels_a))
bh = 0.35
b1 = ax1.barh(y1 - bh/2, inv_recalls, bh, color='#0284C7', edgecolor='#0F172A', linewidth=0.8, label='Invalidated Defect Recall (5 Cases)')
b2 = ax1.barh(y1 + bh/2, ctrl_clears, bh, color='#059669', edgecolor='#0F172A', linewidth=0.8, label='Control Section Clearance (94 Op. Secs / V0=5)')

for idx, (rv, cv) in enumerate(zip(inv_recalls, ctrl_clears)):
    ax1.text(rv + 1.5, idx - bh/2, f'{rv:.0f}%', va='center', ha='left', fontsize=7.6, fontweight='bold', color='#0369A1')
    ax1.text(cv + 1.5, idx + bh/2, f'{cv:.1f}%', va='center', ha='left', fontsize=7.6, fontweight='bold', color='#065F46')

ax1.annotate('Clearance drops 51.1% -> 31.9%\n& UI pairs double (189 -> 377)',
             xy=(ctrl_clears[-1] + 14.0, 6 + bh/2), xytext=(54.0, 6 + bh/2),
             va='center', ha='left',
             arrowprops=dict(arrowstyle='->', color='#DC2626', lw=1.2),
             fontsize=7.1, fontweight='bold', color='#991B1B',
             bbox=dict(boxstyle='round,pad=0.26', facecolor='#FEF2F2', edgecolor='#DC2626', lw=0.9))

ax1.set_yticks(y1)
ax1.set_yticklabels(labels_a, fontsize=8.0)
ax1.invert_yaxis()
ax1.set_xlabel('Percentage (%)')
ax1.set_xlim(0, 118)
ax1.set_title('(A) Tier 3 Ablation: Defect Recall vs. Control Clearance', pad=12)
ax1.grid(axis='x', linestyle='--', alpha=0.42)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.legend(loc='upper center', bbox_to_anchor=(0.45, -0.14), ncol=1, fontsize=7.6, frameon=True, facecolor='#FFFFFF', edgecolor='#CBD5E1')

# --- Panel B: Synchronized Full-PDF Open-Corpus Audit (N = 10 PDFs, V5) ---
ax2 = fig.add_subplot(gs[1])
live_order = [
    ('DAVAO-03/04 Rev. Code (129p)', 'DAVAO-TIER3-03'),
    ('DAVAO-02 Evasco (14p)',        'DAVAO-TIER3-02'),
    ('DAVAO-08 Speed Limits (13p)',  'DAVAO-TIER3-08'),
    ('DAVAO-05 GSIS (7p)',           'DAVAO-TIER3-05'),
    ('DAVAO-07 Anti-Smoking (6p)',   'DAVAO-TIER3-07'),
    ('SC-PHIL-09 Magtajas (2p)',     'SC-PHIL-09'),
    ('DAVAO-06 Mining Ban (3p)',     'DAVAO-TIER3-06'),
    ('SC-PHIL-10 Manila (2p)',       'SC-PHIL-10'),
    ('DAVAO-01 Mosqueda (2p)',       'DAVAO-TIER3-01'),
]

y2 = np.arange(len(live_order))
s1_vals = [max(0.05, tier3_live[cid]['latency_metrics']['stage1_latency_s']) for _, cid in live_order]
s2_vals = [tier3_live[cid]['latency_metrics']['stage2_latency_s'] for _, cid in live_order]
tot_vals = [tier3_live[cid]['audit_latency_s'] for _, cid in live_order]
flags_vals = [tier3_live[cid]['total_conflicts_found'] for _, cid in live_order]
op_vals = [tier3_live[cid]['substantive_sections'] for _, cid in live_order]

ax2.barh(y2, s1_vals, height=0.58, color='#0284C7', edgecolor='#0F172A', linewidth=0.75, label='Stage 1 Sparse CSR + MiniLM (s)')
ax2.barh(y2, s2_vals, left=s1_vals, height=0.58, color='#7C3AED', edgecolor='#0F172A', linewidth=0.75, label='Stage 2 DeBERTa-v3 + XAI (s)')

ax2.set_xscale('log')
ax2.set_xlim(0.15, 65000)
for idx, (tot, fl, op) in enumerate(zip(tot_vals, flags_vals, op_vals)):
    ax2.text(tot * 1.15, idx, f'{tot:.1f}s | {fl} flags / {op} op.',
             va='center', ha='left', fontsize=7.6, fontweight='bold' if fl >= 700 or 'SC-PHIL-09' in live_order[idx][1] else 'normal', color='#0F172A')

ax2.set_yticks(y2)
ax2.set_yticklabels([lbl for lbl, _ in live_order], fontsize=8.0)
ax2.invert_yaxis()
ax2.set_xlabel('Offline Full-PDF CPU Execution Time (s, Log Scale)')
ax2.set_title('(B) Synchronized Full-PDF Open-Corpus Audit (N = 10, V5)', pad=12)
ax2.grid(axis='x', which='both', linestyle='--', alpha=0.40)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.14), ncol=1, fontsize=7.6, frameon=True, facecolor='#FFFFFF', edgecolor='#CBD5E1')

out7 = os.path.join(FIGS_DIR, 'tier3_ablation_and_open_corpus_audit.png')
plt.savefig(out7, dpi=300, bbox_inches='tight')
plt.close(fig)
print('Saved:', out7)

Saved: C:\Users\SHRIMP\Documents\thesis-repo\CS_Undergraduate_Thesis_Template\figs\tier3_ablation_and_open_corpus_audit.png
